In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima.arima import auto_arima


# Carregamento do dataset
dataset = pd.read_csv('datasets_56102_107707_monthly-beer-production-in-austr.csv',  delimiter=',', index_col='Month', parse_dates=True)
serie = dataset['Monthly beer production'].astype(float)
print(dataset)


# 1. Traçando a série temporal
print(f'Serie mensal analisada: {serie.index.min():%m/%Y} a {serie.index.max():%m/%Y}, com {len(serie)} observacoes.')
serie.plot(figsize=(8, 6),
title='Produção mensal de cerveja na Austrália',
xlabel='Data', ylabel='Produção (Megalitros)', x_compat=True)

plt.show()


# 2. Decomposição da série temporal
print('Decomposicao multiplicativa da serie com periodo sazonal anual de 12 meses.')
decomposition = seasonal_decompose(serie, model='multiplicative', period=12)
decomposition.plot()
plt.tight_layout()
plt.show()


# 3. A série possui tendência? Se sim, que tipo?
print('Resposta: tendencia nao linear; a producao cresce ate o fim dos anos 1970 e depois oscila com queda geral.')
decomposition.trend.plot(figsize=(8, 6),
title='Tendência da produção de cerveja',
xlabel='Data', ylabel='Produção (Megalitros)')
plt.show()


# 4. A série possui sazonalidade? Se sim, qual o período que ela acontece?
print('Resposta: sim. A sazonalidade se repete a cada 12 meses, com producao geralmente maior no fim do ano e menor no meio do ano.')
decomposition.seasonal.iloc[:36].plot(figsize=(8, 6),
title='Sazonalidade da produção de cerveja',
xlabel='Data', ylabel='Componente sazonal')

plt.show()

meses = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
         'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

plt.figure(figsize=(8, 6))

for ano in range(1990, 1995):
    valores = np.array(serie[str(ano)])
    plt.plot(meses, valores, label=str(ano))

plt.title('Comportamento mensal da produção')
plt.xlabel('Mês')
plt.ylabel('Produção (Megalitros)')
plt.legend()
plt.show()


# 5. A série apresenta um ciclo? Se sim, por qual razão?
print('Resposta: ha oscilacoes de varios anos, mas nao um ciclo regular de duracao fixa. A causa nao pode ser determinada apenas com esta serie.')
anos = []
medias = []

for ano in range(1956, 1995):
    valores = serie[str(ano)]
    anos.append(ano)
    medias.append(valores.mean())

plt.figure(figsize=(8, 6))
plt.plot(anos, medias)
plt.title('Comportamento da produção no longo prazo')
plt.xlabel('Ano')
plt.ylabel('Produção média anual (Megalitros)')
plt.show()


# 6. Identificação do método mais adequado para previsão
meses_teste = int(len(serie) * 0.2)

conjunto_treinamento = serie.iloc[:-meses_teste]
conjunto_teste = serie.iloc[-meses_teste:]


# Holt-Winters

model = ExponentialSmoothing(endog=conjunto_treinamento, trend='mul', seasonal='mul', seasonal_periods=12).fit()
forecasting_hw = model.forecast(steps=len(conjunto_teste))


# ARIMA
model = auto_arima(y=conjunto_treinamento, m=12, stepwise=True, trace=True, max_p=2, max_q=2, max_P=1, max_Q=1)
print(model)
forecasting_arima = pd.Series(model.predict(n_periods=len(conjunto_teste)))


# Cálculo do MAPE
def MAPE(y_true, y_pred):
  y_true, y_pred = np.array(y_true), np.array(y_pred)
  return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

mape_hw = MAPE(conjunto_teste, forecasting_hw)
mape_arima = MAPE(conjunto_teste, forecasting_arima)

print('MAPE Holt-Winters:', mape_hw)
print('MAPE ARIMA:', mape_arima)
if mape_hw <= mape_arima:
    print(f'Resposta: Holt-Winters foi o mais adequado nesta comparacao (MAPE {mape_hw:.2f}% contra {mape_arima:.2f}% do ARIMA).')
else:
    print(f'Resposta: ARIMA foi o mais adequado nesta comparacao (MAPE {mape_arima:.2f}% contra {mape_hw:.2f}% do Holt-Winters).')



# Comparação dos métodos
plt.figure(figsize=(8, 6))
plt.plot(conjunto_teste.index, np.array(conjunto_teste), label='Real')
plt.plot(conjunto_teste.index, np.array(forecasting_hw), label='Holt-Winters')
plt.plot(conjunto_teste.index, np.array(forecasting_arima), label='ARIMA')
plt.title('Comparação dos métodos de previsão')
plt.xlabel('Data')
plt.ylabel('Produção (Megalitros)')
plt.legend()
plt.show()


# 7. Previsão de valores futuros

if mape_hw <= mape_arima:
    print('Método selecionado: Holt-Winters')

    model = ExponentialSmoothing(endog=serie, trend='mul', seasonal='mul', seasonal_periods=12).fit()
    predictions = model.forecast(steps=12)

else:
    print('Método selecionado: ARIMA')
    model = auto_arima(y=serie, m=12, stepwise=True, trace=True, max_p=2, max_q=2, max_P=1, max_Q=1)
    predictions = pd.Series(model.predict(n_periods=12))

print('Resposta: previsao da producao para os proximos 12 meses:')
print(predictions)
print('O metodo selecionado foi aquele com menor MAPE no conjunto de teste.')

serie['1990-01-01':].plot(figsize=(8, 6), legend=True, label='Histórico')
predictions.plot(legend=True, label='Previsão')

plt.title('Previsão da produção industrial de cerveja')
plt.xlabel('Data')
plt.ylabel('Produção (Megalitros)')
plt.show()



You should consider upgrading via the 'c:\users\mathe\appdata\local\programs\python\python38\python.exe -m pip install --upgrade pip' command.


FileNotFoundError: [Errno 2] No such file or directory: 'monthly-beer-production.csv'